# Claude Managed Agents로 데이터 분석 에이전트 만들기

## 들어가며

어느 팀에나 CSV를 건네받고 "여기서 뭐가 흥미로운지 봐 줘"라는 요청을 받는 사람이 있습니다. 이 쿡북에서는 그 사람 대신 답해 주는 에이전트를 만듭니다. CSV를 업로드하면 인터랙티브 차트가 담긴 서술형 HTML 보고서를 돌려받습니다.

상태를 유지하며 도구를 사용하는 에이전트를 위한 Anthropic의 호스팅 런타임 [Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview)에서 실행합니다. 네 가지 핵심 개념 위에 만들어져 있습니다.

- **에이전트(Agent)**: 모델, 시스템 프롬프트, 도구, MCP 서버, 스킬
- **환경(Environment)**: 설정이 끝난 컨테이너 템플릿(패키지, 네트워크 접근)
- **세션(Session)**: 환경 안에서 실행되는 에이전트 인스턴스로, 특정 작업을 수행하고 산출물을 만듭니다
- **이벤트(Events)**: 여러분의 애플리케이션과 에이전트가 주고받는 메시지(사용자 턴, 도구 결과, 상태 업데이트)

에이전트와 환경이 합쳐지면 세션이 됩니다. 여기에 데이터를 리소스로 붙인 뒤, 이벤트를 보내고 스트림을 읽으며 구동합니다.

샌드박스, 도구 실행, 컨텍스트 관리는 Anthropic이 대신 처리합니다. 에이전트 루프와 배포를 완전히 직접 제어해야 한다면 [Claude Agent SDK](https://platform.claude.com/docs/en/api/agent-sdk/overview)를 사용하세요.

### 배울 내용

이 쿡북을 마치면 다음을 할 수 있습니다.

- 데이터 분석을 위한 재사용 가능한 환경과 에이전트 준비하기
- 에이전트에 다룰 데이터셋 건네기
- 에이전트가 실행되는 동안 진행 상황 지켜보기
- 보고서와 그 밖의 산출 파일 내려받기

### 사전 준비

- Python 3.11 이상
- [콘솔](https://platform.claude.com/settings/keys)에서 발급받아 `ANTHROPIC_API_KEY`로 설정한 Anthropic API 키

의존성 설치:

In [10]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv

In [11]:
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv, set_key

load_dotenv()
client = Anthropic()
MODEL = "claude-sonnet-4-6"

## 1. 환경 만들기

**환경**은 재사용 가능한 컨테이너 명세입니다. 여기서 `pandas`와 `plotly`를 선언해 두면 모든 세션이 이들이 미리 설치된 상태로 시작하므로, 에이전트가 `pip install`부터 하지 않고 곧바로 분석을 시작할 수 있습니다.

여기서는 에이전트가 CDN에서 plotly를 불러올 수 있도록 네트워킹을 `unrestricted`로 두었습니다. 다만 이렇게 하면 인터넷 어디에나 접근할 수 있으므로, 프로덕션에서는 [호스트 허용 목록](https://platform.claude.com/docs/en/managed-agents/environments)을 사용하세요.

In [12]:
env = client.beta.environments.create(
    name="cookbook-data-analyst-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
        "packages": {
            "type": "packages",
            "pip": ["pandas", "plotly"],
        },
    },
)

## 2. 에이전트 만들기

**에이전트**는 모델에 시스템 프롬프트와 도구 세트를 짝지은 것입니다. 출력 품질의 대부분은 시스템 프롬프트에서 나옵니다. 여기서는 서술형 구조, 구체적인 수치로 뒷받침된 발견, 그리고 여러 plotly 차트를 하나의 HTML 파일에 담는 올바른 패턴을 요구합니다.

[`agent_toolset_20260401`](https://platform.claude.com/docs/en/managed-agents/tools)은 여덟 가지 도구를 제공합니다. `bash`, `read`, `write`, `edit`, `glob`, `grep`, `web_fetch`, `web_search`입니다. 여기서는 모두 `always_allow`로 실행하되, 이번 분석은 오프라인이므로 웹 관련 두 도구는 비활성화했습니다.

In [13]:
ANALYST_SYSTEM_PROMPT = """\
You are a senior data analyst producing a publication-quality report.

## Style
- Professional and precise. Let the data speak with concrete numbers.
- Short paragraphs (2-3 sentences) between charts.
- Lead with the most actionable finding.

## Execution
- Write .py scripts and run them with `python3 script.py`.
- Sample large tables (`nrows=` / `.sample()`) instead of loading everything.
- Sanity-check key metrics before building narrative around them.

## Charts
- Build each chart as its own `go.Figure()`, embed with
  `fig.to_html(include_plotlyjs=False, full_html=False)`, and load plotly
  from the CDN once in <head>.
- Always set `marker_color` and `template='simple_white'`.

## Output
Write a single self-contained `report.html` to /mnt/session/outputs/
with inline CSS, 3+ embedded plotly charts, and a closing section of
actionable recommendations. Confirm "Saved: report.html" when done.
"""

agent = client.beta.agents.create(
    name="cookbook-data-analyst",
    model=MODEL,
    system=ANALYST_SYSTEM_PROMPT,
    tools=[
        {
            "type": "agent_toolset_20260401",
            # default_config applies to every tool in the set;
            # entries in configs override specific tools.
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        }
    ],
)

## 3. 데이터셋 업로드

포함된 샘플 CSV는 50행이라 분석이 몇 분 안에 끝납니다. 원하는 CSV(또는 CSV들을 묶은 zip)로 바꿔도 되며, 나머지 흐름은 동일합니다.

In [14]:
DATA_PATH = Path("example_data/data_analyst_agent/sales_data.csv")

with DATA_PATH.open("rb") as f:
    dataset = client.beta.files.upload(file=(DATA_PATH.name, f, "text/csv"))

print(f"Uploaded {DATA_PATH.name} ({dataset.size_bytes} bytes) as {dataset.id}")

Uploaded sales_data.csv (2833 bytes) as file_011CZqKKomgL45BLo2J8K5X9


## 4. 세션 만들고 작업 보내기

**세션**은 에이전트를 환경 및 마운트된 파일과 묶어 줍니다. `{"type": "agent", "id": ..., "version": ...}`을 전달하면 위에서 만든 버전 지정 에이전트를 재사용합니다. `resources`는 업로드한 파일을 컨테이너 안의 지정된 절대 경로에 마운트합니다.

세션을 만든 뒤 작업 내용을 담은 `user.message` 이벤트를 보냅니다. 에이전트가 곧바로 일을 시작합니다.

In [15]:
MOUNT_PATH = f"/mnt/session/uploads/{DATA_PATH.name}"

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[{"type": "file", "file_id": dataset.id, "mount_path": MOUNT_PATH}],
    title="Sales analysis",
)

ANALYSIS_PROMPT = f"""\
Analyze the e-commerce orders in {MOUNT_PATH}.

Columns: order_id, customer_id, product, category, price, quantity,
order_date, region.

Focus on revenue by category and region, repeat-customer behavior, and
one surprising pattern. Produce report.html per your system instructions.
"""

client.beta.sessions.events.send(
    session.id,
    events=[
        {"type": "user.message", "content": [{"type": "text", "text": ANALYSIS_PROMPT}]},
    ],
)
print(f"Session {session.id} running")

Session sesn_011CZqKKqyPRph9J5khKLn6u running


## 5. 실행 스트리밍하기

[콘솔](https://platform.claude.com/)의 **Sessions**에서 세션을 열면 모든 이벤트, 도구 호출, 토큰 수를 실시간으로 볼 수 있습니다:

<img src="https://raw.githubusercontent.com/anthropics/claude-cookbooks/main/managed_agents/example_data/data_analyst_agent/console_session.png" alt="콘솔에 표시된 세션 추적" width="700" />

아래 헬퍼는 같은 이벤트 스트림을 따라가며 `agent.message` 텍스트와 `agent.tool_use` 호출을 도착하는 대로 출력하고, `session.status_idle`에서 반환합니다.

In [ ]:
def wait_for_idle(session_id: str) -> None:
    for ev in client.beta.sessions.events.stream(session_id):
        t = ev.type
        if t == "agent.message":
            for block in ev.content:
                if block.type == "text":
                    text = block.text
                    print(text[:300] + ("..." if len(text) > 300 else ""))
        elif t in ("agent.tool_use", "agent.mcp_tool_use"):
            print(f"  [{ev.name}]")
        elif t == "session.status_idle":
            return
        elif t == "session.status_terminated":
            raise RuntimeError(
                "Session terminated before going idle. "
                f"Trace: https://platform.claude.com/sessions/{session_id}"
            )


wait_for_idle(session.id)

## 6. 보고서 가져오기

에이전트가 `/mnt/session/outputs/`에 쓴 것은 모두 보존되며 `scope_id=<session_id>`로 Files API를 통해 노출됩니다. 컨테이너의 다른 위치에 쓴 파일은 보존되지 않습니다.

[Files API](https://platform.claude.com/docs/en/api/beta/files/list)는 베타 단계의 별도 기능이므로, 여기서 `scope_id`를 쓰려면 Managed Agents 베타 헤더도 함께 전달해야 합니다.

In [17]:
outputs = client.beta.files.list(scope_id=session.id, betas=["managed-agents-2026-04-01"])
for f in outputs.data:
    print(f.filename, f.size_bytes)

# The list also includes the mounted input CSV; pick out the report.
report = next((f for f in outputs.data if f.filename == "report.html"), None)
if report is None:
    raise RuntimeError(f"report.html not found. Files: {[f.filename for f in outputs.data]}")
content = client.beta.files.download(report.id)
Path("report.html").write_bytes(content.read())
print("Downloaded report.html")

report.html 53728
sales_data.csv 2833
Downloaded report.html


## 7. 정리와 다음 단계

에이전트와 환경은 한 번 만들어 여러 실행에 걸쳐 재사용하고, 대화마다 새 세션을 만듭니다. 보고서를 받았으니 이 세션을 아카이브해 컨테이너를 반환하세요. 아래 코드는 [`slack_data_bot.ipynb`](slack_data_bot.ipynb)가 이 에이전트로 새 세션을 시작할 수 있도록 에이전트와 환경 ID를 `.env`에 저장합니다.

> **주의:** 다음 셀을 실행하기 전에 `.env`가 `.gitignore`에 있는지 확인하세요. 절대 커밋하지 마세요.

In [18]:
client.beta.sessions.archive(session.id)

set_key(".env", "ANALYST_ENV_ID", env.id)
set_key(".env", "ANALYST_AGENT_ID", agent.id)
set_key(".env", "ANALYST_AGENT_VERSION", str(agent.version))
print("Saved ANALYST_ENV_ID, ANALYST_AGENT_ID, ANALYST_AGENT_VERSION to .env")

Saved ANALYST_ENV_ID, ANALYST_AGENT_ID, ANALYST_AGENT_VERSION to .env


데이터 분석 에이전트를 처음부터 끝까지 만들어 실행해 봤습니다. 재사용 가능한 환경과 에이전트, CSV를 마운트한 세션, 실시간 이벤트 스트림, 그리고 내려받은 HTML 보고서까지입니다.

여기서부터는:

- 내려받은 `report.html`을 열어 서술과 차트를 확인해 보세요.
- [콘솔](https://platform.claude.com/)에서 세션을 열어 토큰 사용량과 전체 이벤트 로그를 살펴보세요.
- [`slack_data_bot.ipynb`](slack_data_bot.ipynb)로 넘어가 이 에이전트를 Slack에서 구동해 보세요.